# Code mẫu — Support Vector Machine

Trước khi gọi thư viện, ta tự tính ba con số cốt lõi của SVM: score, khoảng cách tới đường biên, và hinge loss. Sau đó mới xem support vector, kernel tạo biên cong, và ảnh hưởng của đơn vị đo. Suốt trang này, score của SVM luôn được tách khỏi xác suất.

## Ví dụ 1: Score, khoảng cách và hinge loss

Cho `w = (3, 4)` và `b = −10`, nên độ dài của `w` là 5. Với ba điểm đều thuộc lớp +1, ta tính score có dấu, khoảng cách hình học tới đường biên, và hinge loss.

In [ ]:
import numpy as np

w = np.array([3., 4.])
b = -10.
points = np.array([[2., 1.], [3., 2.], [2., 1.1]])
y_signed = np.array([1., 1., 1.])
scores = points @ w + b
distances = np.abs(scores) / np.linalg.norm(w)
hinge = np.maximum(0., 1. - y_signed * scores)
print("score:", scores.round(3).tolist())
print("khoảng cách:", distances.round(3).tolist())
print("hinge:", hinge.round(3).tolist())
assert np.allclose(scores, [0., 7., 0.4])
assert np.allclose(distances, [0., 1.4, 0.08])
assert np.allclose(hinge, [1., 0., 0.6])

score: [0.0, 7.0, 0.4]
khoảng cách: [0.0, 1.4, 0.08]
hinge: [1.0, 0.0, 0.6]


Điểm đầu nằm ngay trên đường biên, nên score bằng 0 và hinge bằng 1. Điểm thứ hai có score 7, nằm xa ngoài lề, nên hinge bằng 0. Điểm thứ ba đáng chú ý nhất. Score của nó dương, nên nó nằm đúng phía. Nhưng score mới là 0.4, chưa tới 1, nên nó vẫn ở trong lề và vẫn bị phạt 0.6.

Score 7 không có nghĩa là xác suất 70%. Muốn ra khoảng cách thật tới đường biên, phải chia score cho độ dài của `w`: 7 chia 5 bằng 1.4.

Một lỗi rất dễ mắc mà không ai báo là dùng nhãn 0 và 1 trong công thức hinge. Hinge cần nhãn −1 và +1. Nếu nhãn là 0, tích của nhãn với score luôn bằng 0, bất kể score là bao nhiêu.

In [ ]:
scores_check = np.array([0.4, -0.2])
y01 = np.array([1., 0.])
y_signed_check = np.array([1., -1.])
wrong = np.maximum(0., 1. - y01 * scores_check)
correct = np.maximum(0., 1. - y_signed_check * scores_check)
print("hinge dùng 0/1:", wrong.round(3).tolist())
print("hinge đổi sang -1/+1:", correct.round(3).tolist())
assert np.allclose(wrong, [0.6, 1.0])
assert np.allclose(correct, [0.6, 0.8])

hinge dùng 0/1: [0.6, 1.0]
hinge đổi sang -1/+1: [0.6, 0.8]


Mẫu thứ hai có score −0.2, tức đã nghiêng đúng về phía lớp âm. Với nhãn −1, hinge là 0.8. Với nhãn 0, hinge thành 1, như thể model chưa học được gì về mẫu này. Cách sửa là đổi nhãn sang −1 và +1 trước khi nhân với score. Nếu một điểm có score đúng bằng 0, cần một luật phá hòa để quyết định nó thuộc lớp nào.

## Ví dụ 2: Biên thẳng và biên cong nhờ kernel

Dữ liệu `make_moons` gồm hai hình lưỡi liềm lồng vào nhau, nên một đường thẳng khó tách tốt. Trên cùng một cách chia, ta fit SVM với kernel linear và kernel RBF, cả hai đều có chuẩn hóa trong Pipeline. Sau đó đọc accuracy validation, số support vector, và score của ba điểm đầu.

In [ ]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

X, y = make_moons(n_samples=180, noise=0.22, random_state=4)
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=2
)
for kernel in ["linear", "rbf"]:
    model = make_pipeline(
        StandardScaler(), SVC(kernel=kernel, C=1., gamma="scale")
    ).fit(X_train, y_train)
    svc = model[-1]
    score = model.decision_function(X_valid[:3])
    print(kernel, "valid:", round(accuracy_score(y_valid, model.predict(X_valid)), 3),
          "support:", int(svc.n_support_.sum()),
          "score 3 điểm:", np.round(score, 3).tolist())
    assert svc.n_support_.sum() > 0

linear valid: 0.833 support: 57 score 3 điểm: [-1.203, -1.906, 0.175]
rbf valid: 0.981 support: 59 score 3 điểm: [-1.407, -1.316, -0.42]


Kernel RBF cho đường biên được uốn cong, và accuracy validation tăng từ 0.833 lên 0.981. Hãy nhìn điểm thứ ba: model linear cho score dương, còn RBF cho score âm. Hai model đặt điểm này ở hai phía khác nhau của đường biên.

Support vector là những mẫu train nằm trên lề hoặc vi phạm lề. Chỉ chúng quyết định hình dạng đường biên, còn các mẫu nằm xa thì không ảnh hưởng. `decision_function` chỉ trả về score có dấu, không phải xác suất. Muốn có xác suất, phải hiệu chuẩn bằng một quy trình validation riêng.

Kernel cho phép làm việc trong một không gian nhiều chiều hơn mà không cần dựng không gian ấy ra. Với kernel đa thức bậc hai, ta kiểm được điều này bằng tay. Tích vô hướng sau khi đưa hai vector hai chiều lên ba chiều bằng đúng bình phương tích vô hướng ban đầu.

In [ ]:
x_kernel = np.array([1., 2.])
z_kernel = np.array([3., 4.])

def phi2(v):
    a, b = v
    return np.array([a * a, np.sqrt(2.) * a * b, b * b])

expanded = phi2(x_kernel) @ phi2(z_kernel)
kernel_value = (x_kernel @ z_kernel) ** 2
print("dot trong không gian mở rộng:", round(float(expanded), 3))
print("kernel (x · z)^2:", round(float(kernel_value), 3))
assert np.isclose(expanded, kernel_value)

dot trong không gian mở rộng: 121.0
kernel (x · z)^2: 121.0


Hai cách đều cho 121. Cách thứ hai không cần dựng vector ba chiều nào, chỉ cần một tích vô hướng rồi bình phương lên. Khi không gian mở rộng có hàng nghìn chiều, hoặc vô hạn chiều như với RBF, đây là cách duy nhất làm được.

## Ví dụ 3: Chuẩn hóa và dò gamma trên validation

Ta nhân cột thứ hai của `make_moons` lên 100 lần, giống như hai cột đo bằng hai đơn vị lệch nhau. So một SVC RBF dùng dữ liệu thô với một Pipeline có chuẩn hóa. Sau đó thử ba giá trị `gamma` trên validation. Chỉ khi đã chọn xong gamma, ta mới chấm test.

In [ ]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

X, y = make_moons(n_samples=240, noise=0.24, random_state=6)
X = X * np.array([1., 100.])
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=7
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=3
)

raw = SVC(kernel="rbf", C=2., gamma=1.).fit(X_train, y_train)
scaled = make_pipeline(
    StandardScaler(), SVC(kernel="rbf", C=2., gamma=1.)
).fit(X_train, y_train)
print("RBF thô valid:", round(accuracy_score(y_valid, raw.predict(X_valid)), 3))
print("RBF có scaler valid:", round(accuracy_score(y_valid, scaled.predict(X_valid)), 3))

gamma_rows = []
for gamma in [0.1, 1., 10.]:
    candidate = make_pipeline(
        StandardScaler(), SVC(kernel="rbf", C=2., gamma=gamma)
    ).fit(X_train, y_train)
    valid_accuracy = accuracy_score(y_valid, candidate.predict(X_valid))
    gamma_rows.append((gamma, float(valid_accuracy), int(candidate[-1].n_support_.sum())))
    print("gamma =", gamma, "valid:", round(float(valid_accuracy), 3),
          "support:", int(candidate[-1].n_support_.sum()))

best_gamma, _, _ = max(gamma_rows, key=lambda row: row[1])
chosen = make_pipeline(
    StandardScaler(), SVC(kernel="rbf", C=2., gamma=best_gamma)
).fit(X_dev, y_dev)
test_accuracy = accuracy_score(y_test, chosen.predict(X_test))
print("gamma chọn:", best_gamma, "test sau khi khóa:", round(float(test_accuracy), 3))
assert best_gamma in {0.1, 1.0, 10.0}

RBF thô valid: 0.771
RBF có scaler valid: 0.958
gamma = 0.1 valid: 0.896 support: 58
gamma = 1.0 valid: 0.958 support: 40
gamma = 10.0 valid: 0.938 support: 92
gamma chọn: 1.0 test sau khi khóa: 0.938


Không chuẩn hóa thì khoảng cách gần như chỉ do cột thứ hai quyết định, và accuracy chỉ còn 0.771. Có scaler, cùng gamma ấy cho 0.958. Gamma đo độ rộng của vùng ảnh hưởng quanh mỗi điểm theo đơn vị của khoảng cách, nên đổi thang đo thì một con số gamma đã mang nghĩa khác.

Đọc ba dòng gamma cùng với số support vector. Gamma 0.1 cho biên quá trơn, chưa uốn kịp theo hai lưỡi liềm. Gamma 10 cho biên uốn quá kỹ, và cần tới 92 support vector, gấp đôi so với gamma 1. Gamma 1 vừa đạt điểm cao nhất, vừa cần ít support vector nhất. Lưới `0.1, 1, 10` tăng theo bội số 10, là cách dò thường dùng cho gamma và C. Nếu thử thêm nhiều cấu hình nữa, validation vẫn đang bị dùng để chọn, nên test phải giữ kín tới lần báo cáo cuối.